# 02. 데이터 전처리 파이프라인
**데이터셋**: Cardiovascular Disease Dataset (Kaggle, 70,000명)  
**목적**: XGBoost 심혈관 위험도 예측 모델 학습을 위한 전처리  
**참고 논문**: XGBoost 기반 K-Means 군집 분석을 활용한 심장질환 예측 (한국정보전자통신기술학회, 2025)

In [ ]:
# ── 셀 1: 라이브러리 import
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# ── 셀 2: 데이터 로드
df = pd.read_csv('/Users/admin/cardiovascular_ml/data/cardio_train.csv', sep=';')
print(f"원본 데이터: {df.shape}")
print(f"컬럼 목록: {df.columns.tolist()}")

In [ ]:
# ── 셀 3: 나이 변환 (일 → 세)
# 원본 데이터의 age는 일(days) 단위 → 세(years) 단위로 변환 필요
df['age'] = (df['age'] / 365).astype(int)
print(f"변환 후 나이 범위: {df['age'].min()}세 ~ {df['age'].max()}세")

In [ ]:
# ── 셀 4: BMI 파생 변수 생성
# BMI = 심혈관 질환 위험 요인 (미국임상내분비학회 2016 기준 BMI 25+ 위험)
# 데이터셋에 BMI 없어서 height, weight로 직접 계산
df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)
print(f"BMI 범위: {df['bmi'].min():.2f} ~ {df['bmi'].max():.2f}")

In [ ]:
# ── 셀 5: 이상치 제거 (의학적 정상 범위 기준)
# height 100~250 기준으로 먼저 걸렀으나
# height > 200 이상치 103개 확인 후 기준 강화
before = len(df)

df = df[
    (df['ap_hi'] >= 80)   & (df['ap_hi'] <= 250) &    # 수축기 혈압 정상 범위
    (df['ap_lo'] >= 50)   & (df['ap_lo'] <= 200) &    # 이완기 혈압 정상 범위
    (df['height'] >= 140) & (df['height'] <= 210) &   # 키 정상 범위
    (df['weight'] >= 40)  & (df['weight'] <= 180) &   # 몸무게 정상 범위
    (df['bmi'] >= 10)     & (df['bmi'] <= 60)         # BMI 정상 범위
]

after = len(df)
print(f"이상치 제거: {before}명 → {after}명 ({before - after}명 제거)")

In [ ]:
# ── 셀 6: 인코딩 확인
# cholesterol, gluc, gender 이미 숫자형 → 별도 인코딩 불필요
print(f"cholesterol: {sorted(df['cholesterol'].unique())} → 1(정상) 2(경계) 3(높음) Ordinal 유지")
print(f"gluc:        {sorted(df['gluc'].unique())} → 1(정상) 2(경계) 3(높음) Ordinal 유지")
print(f"gender:      {sorted(df['gender'].unique())} → 1(여성) 2(남성) 그대로 유지")

In [ ]:
# ── 셀 7: 피처 / 타겟 분리
X = df.drop(columns=['id', 'cardio'])
y = df['cardio']
print(f"피처 수: {X.shape[1]}개")
print(f"피처 목록: {X.columns.tolist()}")
print(f"타겟 분포:\n{y.value_counts()}")

In [ ]:
# ── 셀 8: train/test 분리 (8:2)
# stratify=y → 클래스 비율 유지
# random_state=42 → 재현 가능한 결과 (참고 논문 동일 설정)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)
print(f"훈련 데이터: {X_train.shape}")
print(f"테스트 데이터: {X_test.shape}")
print(f"\n훈련 타겟 비율:\n{y_train.value_counts(normalize=True).round(4)}")
print(f"\n테스트 타겟 비율:\n{y_test.value_counts(normalize=True).round(4)}")

In [ ]:
# ── 셀 9: 정규화 (연속형 피처만)
# XGBoost는 정규화 필수 아니지만 피처 스케일 통일로 성능 향상
# 주의: fit은 train에만! test에는 transform만 적용 (데이터 누수 방지)
scaler = StandardScaler()
scale_cols = ['age', 'height', 'weight', 'ap_hi', 'ap_lo', 'bmi']

X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

print("정규화 후 훈련 데이터 통계 (mean≈0, std≈1 이면 정상):")
print(X_train[scale_cols].describe().round(2))

In [ ]:
# ── 셀 10: 전처리 완료 요약
print("=" * 40)
print("전처리 완료 요약")
print("=" * 40)
print(f"원본 데이터:   70,000명")
print(f"이상치 제거 후: {after}명 ({before - after}명 제거)")
print(f"훈련 데이터:   {X_train.shape[0]}명")
print(f"테스트 데이터: {X_test.shape[0]}명")
print(f"피처 수:       {X_train.shape[1]}개")
print(f"피처 목록:     {X_train.columns.tolist()}")
print("=" * 40)
print("전처리 단계 요약")
print("=" * 40)
print("1. 나이 변환:    일(days) → 세(years)")
print("2. BMI 생성:     height, weight로 파생 변수 계산")
print("3. 이상치 제거:  의학적 정상 범위 기준")
print("4. 인코딩:       숫자형 그대로 유지 (별도 인코딩 불필요)")
print("5. 데이터 분리:  train 80% / test 20% (stratify, random_state=42)")
print("6. 정규화:       연속형 피처 StandardScaler (데이터 누수 방지)")